In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from gpt_model2 import GPTModel2
from gpt_model1 import TokenDataset
import tiktoken
import requests
from torchinfo import summary

In [2]:
# tokenize the text
text = requests.get('https://www.gutenberg.org/files/35/35-0.txt').text
tokenizer = tiktoken.get_encoding('cl100k_base')

# text needs to be pytorch tensors
tokens = tokenizer.encode(text)
print(f'Variable "tokens" is type {type(tokens)}')

# convert to pytorch
tmTokens = torch.tensor( tokens )
print(f'Variable "tmTokens" is type {type(tmTokens)} and has {len(tmTokens)}')
print(tmTokens.shape)

Variable "tokens" is type <class 'list'>
Variable "tmTokens" is type <class 'torch.Tensor'> and has 43053
torch.Size([43053])


In [3]:
# Hyper parameters
seq_len = 8 # aka context length
stride = 2
n_vocab = tokenizer.n_vocab
print("Vocab size = ", n_vocab)

# model hyperparameters
embed_dim = 2**6 # 64

batch_size = 5

Vocab size =  100277


In [4]:
token_dataset = TokenDataset(tokenizer, text, seq_len, stride)
# print(len(token_dataset))
token_dataset[4]

dataloader = DataLoader(
                token_dataset,
                batch_size = batch_size,
                shuffle    = True,
                drop_last  = True
            )

# let's have a look at the indices
X,y = next(iter(dataloader))
print("X shape ", X.shape)
print("y shape ", y.shape)
print(tokenizer.decode(X.detach().numpy()[0]))
print(tokenizer.decode(y.detach().numpy()[0]))

X shape  torch.Size([5, 8])
y shape  torch.Size([5, 8])
 from the clutches of the Morlocks
 the clutches of the Morlocks,


In [5]:
model2 = GPTModel2(n_vocab, embed_dim, seq_len)
print(summary(model2))

Layer (type:depth-idx)                   Param #
GPTModel2                                --
├─Embedding: 1-1                         6,417,728
├─Embedding: 1-2                         512
├─GELU: 1-3                              --
├─LayerNorm: 1-4                         128
├─Linear: 1-5                            6,417,728
Total params: 12,836,096
Trainable params: 12,836,096
Non-trainable params: 0


In [6]:
m_out = model2(X)
print("X shape", X.shape)
print("y shape", y.shape)
print("Model out shape", m_out.shape)
print(m_out[:, -1, :].shape)

X shape torch.Size([5, 8])
y shape torch.Size([5, 8])
Model out shape torch.Size([5, 8, 100277])
torch.Size([5, 100277])


In [ ]:
# calc loss for first batch from the model output
fist_batch_logits = m_out[0, :, :] #(ontext_size, vocab_size) => (8, 100277)
print("First batch logits shape = ", fist_batch_logits.shape)
loss = F.cross_entropy(fist_batch_logits, y[0, :]) # y[0, :] => (8,)
print("Cross entropy loss = ", loss)

# manual loss calculation
sf_probs = fist_batch_logits.exp() / torch.sum(fist_batch_logits.exp(), dim=1, keepdim=True)
print("Softmax probs shape = ", sf_probs.shape)
log_sf = torch.log(sf_probs)
nll_loss = -log_sf[torch.arange(8), y[0, :]].mean()
print("NLL loss = ", nll_loss.detach().item())

print(m_out.view(-1, n_vocab).shape) # (batch_size * context_size, vocab_size)
print(y.view(-1).shape) # (batch_size * context_size,)
loss_f = F.cross_entropy(m_out.view(-1, n_vocab), y.view(-1))
print("Cross entropy loss Full = ", loss_f.detach().item())




First batch logits shape =  torch.Size([8, 100277])
Cross entropy loss =  tensor(11.4413, grad_fn=<NllLossBackward0>)
Softmax probs shape =  torch.Size([8, 100277])
NLL loss =  11.441316604614258
torch.Size([40, 100277])
torch.Size([40])
Cross entropy loss Full =  11.740896224975586


In [22]:
# generate text
# print(X.shape)
# print(X[:, -seq_len:].shape)
o = model2.generate(X, gen_len=10)
print(o.shape)
print(tokenizer.decode(o[0].tolist()))
print(tokenizer.decode(o[1].tolist()))
print(tokenizer.decode(o[2].tolist()))

torch.Size([5, 18])
 from the clutches of the Morlocks seine.random.rd GameObject centrif Specifiesdf uninitialized infraredbreadcrumb
 The eyes were large and mild; andcompound hotel plywood-economic restrainedconcert decorators Gunn glands investment
 in his hand was a glittering metallicext zestFMonetJAIncrement오decexclusive Å
